## Cleaning the General Payments Dataset

This script takes in the raw file from the CMS_General_Payments_2021_2023_raw.csv.

**Data Cleaning Steps:**
- Change all data to columns to snake case
- Use the accompanying data dictionary to change the data types of specified columns
- Filter dataset to remove NAs in total_amount_of_payment_us_dollars
- Remove numbers after the dash in the zipcode column
- Drop unrelated columns
- Filter dataset to remove NAs in covered_recipient_npi
- Clean text columns
- Filter out all non-physician practitioners
- Backfill product names

**Feature Engineering:**
- Add fraud-oriented features: `is_weekend_payment`, `is_q4_payment`, `is_travel_payment`, `is_consulting_payment`, `is_speaker_payment`, `is_ownership_payment`, `is_charity_payment`, `is_third_party_payment`
- Add payment timing features: `payment_month`, `payment_quarter`, `payment_to_publication_lag_days`
- Add DME/medical supply categorization: 19 binary columns for high-risk DME categories (e.g., `dme_glucose_monitors`, `dme_urological_supplies`, `dme_surgical_dressings`, etc.)
- Attach LEIE-derived `target` labels at the transaction level

**Split the Data into Different CSV Dim Tables:**

Split the general payments data into domain specific dimensions. Including manufacturing, provider info, products info, and then the general fact records table.

*Returns these CSVs:*
- general_payments-sliced_clean.csv (Record level remaining fact data with product indicators, fraud features, DME flags, and target)
- general_payments_products_clean.csv (Product information, one product ID (PDI) per Row, keeping only most recent program year info)
- general_payments_providers_clean.csv (NPI level data and provider info, one NPI per Row, keeping only most recent program year info)
- general_payments_manufacturers_clean.csv (Manufacturer dimension data)
- cms_general_payments_anomaly_ready.csv (Numeric features prepared for anomaly detection modeling)

**Return an intact and whole cleaned general payments dataset. This data is before splitting into dims.**

In [1]:
# Packages
import numpy as np
import pandas as pd
import glob

In [2]:
import re

def to_snake_case(name: str) -> str:
    # Add underscore between lower-to-upper transitions
    name = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
    name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name)

    # Replace non-alphanumeric with underscores
    name = re.sub(r'[^0-9a-zA-Z]+', '_', name)

    # Remove leading/trailing underscores and lowercase
    return name.strip("_").lower()

### Clean the General Payments Raw File

In [3]:
# Read in General Payments CSV from file path
df = pd.read_csv("/dsa/groups/casestudycf25/team02/bronze/CMS_General_Payments_2021_2023_raw.csv", dtype=str)

df = df.rename(columns={col: to_snake_case(col) for col in df.columns})

df.head(5)

,change_type,covered_recipient_type,teaching_hospital_ccn,teaching_hospital_id,teaching_hospital_name,covered_recipient_profile_id,covered_recipient_npi,covered_recipient_first_name,covered_recipient_middle_name,covered_recipient_last_name,...,associated_drug_or_biological_ndc_4,associated_device_or_medical_supply_pdi_4,covered_or_noncovered_indicator_5,indicate_drug_or_biological_or_device_or_medical_supply_5,product_category_or_therapeutic_area_5,name_of_drug_or_biological_or_device_or_medical_supply_5,associated_drug_or_biological_ndc_5,associated_device_or_medical_supply_pdi_5,program_year,payment_publication_date
0,UNCHANGED,Covered Recipient Physician,NaN,NaN,NaN,294069,1497734156,CARROLL,NaN,JONES,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,06/30/2025
1,UNCHANGED,Covered Recipient Physician,NaN,NaN,NaN,294069,1497734156,CARROLL,NaN,JONES,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,06/30/2025
2,UNCHANGED,Covered Recipient Physician,NaN,NaN,NaN,294069,1497734156,CARROLL,NaN,JONES,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,06/30/2025
3,UNCHANGED,Covered Recipient Physician,NaN,NaN,NaN,294069,1497734156,CARROLL,NaN,JONES,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,06/30/2025
4,UNCHANGED,Covered Recipient Physician,NaN,NaN,NaN,11322702,1932840923,CLAYTON,D,FOSTER,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,06/30/2025


In [4]:
print(f"Original Dataset Shape: {df.shape}")

Original Dataset Shape: (7917764, 91)


In [5]:
df.columns

Index(['change_type', 'covered_recipient_type', 'teaching_hospital_ccn',
       'teaching_hospital_id', 'teaching_hospital_name',
       'covered_recipient_profile_id', 'covered_recipient_npi',
       'covered_recipient_first_name', 'covered_recipient_middle_name',
       'covered_recipient_last_name', 'covered_recipient_name_suffix',
       'recipient_primary_business_street_address_line1',
       'recipient_primary_business_street_address_line2', 'recipient_city',
       'recipient_state', 'recipient_zip_code', 'recipient_country',
       'recipient_province', 'recipient_postal_code',
       'covered_recipient_primary_type_1', 'covered_recipient_primary_type_2',
       'covered_recipient_primary_type_3', 'covered_recipient_primary_type_4',
       'covered_recipient_primary_type_5', 'covered_recipient_primary_type_6',
       'covered_recipient_specialty_1', 'covered_recipient_specialty_2',
       'covered_recipient_specialty_3', 'covered_recipient_specialty_4',
       'covered_recipie

In [6]:
df.recipient_state.unique()

array(['NC', 'MI', 'SC', 'PA', 'IL', 'TX', 'CA', 'IN', 'OH', 'NY', 'WA',
       'TN', 'FL', 'CT', 'UT', 'WI', 'AZ', 'KY', 'IA', 'OR', 'GA', 'ND',
       'NJ', 'MN', 'RI', 'MA', 'AK', 'ID', 'MO', 'NV', 'AE', 'OK', 'SD',
       'KS', 'VA', 'HI', 'MD', 'LA', 'CO', 'NM', 'NE', 'ME', 'MS', 'DC',
       'AL', 'NH', 'MT', 'WV', 'AR', 'DE', 'WY', 'GU', 'PR', 'VT', 'AP',
       nan, 'VI', 'MP', 'AS', 'AA'], dtype=object)

In [7]:
################################################################
# FILTER THE GENERAL PAYMENTS DATASET BY STATES OF INTEREST
###################################################################
df = df[df["recipient_state"].isin(['MO', 'IL', 'IA', 'OK', 'AR', 'TN', 'KS', 'NE', 'KY'])].copy()

In [8]:
print(f"Filtered for States Dataset Shape: {df.shape}")

Filtered for States Dataset Shape: (1146735, 91)


In [9]:
# All other columns strings per data dictionary
# program_year -> integer
# payment_publication_date -> date
# date_of_payment -> date
# total_amount_of_payment_usdollars -> number
# number_of_payments_included_in_total_amount -> integer
# covered_recipient_npi -> integer

special_types = {
    "program_year": "Int64",   # Pandas nullable integer
    "number_of_payments_included_in_total_amount": "Int64",
    "covered_recipient_npi": "Int64",
    "total_amount_of_payment_us_dollars": "float64",
    "payment_publication_date": "datetime64[ns]",
    "date_of_payment": "datetime64[ns]",
}

In [10]:
for col, dtype in special_types.items():
    if dtype == "datetime64[ns]":
        df[col] = pd.to_datetime(df[col], errors = "coerce")

    elif dtype == "Int64":
        df[col] = pd.to_numeric(df[col], errors = "coerce").astype("Int64")

    elif dtype == "float64":
        df[col] = pd.to_numeric(df[col], errors = "coerce")

    else:
        df[col] = df[col].astype(dtype)

In [11]:
print(df.dtypes)

change_type                                                         object
covered_recipient_type                                              object
teaching_hospital_ccn                                               object
teaching_hospital_id                                                object
teaching_hospital_name                                              object
                                                                 ...      
name_of_drug_or_biological_or_device_or_medical_supply_5            object
associated_drug_or_biological_ndc_5                                 object
associated_device_or_medical_supply_pdi_5                           object
program_year                                                         Int64
payment_publication_date                                    datetime64[ns]
Length: 91, dtype: object


In [12]:
# Remove numbers after the dash in the zipcode column. Example 37212-4354 -> 37212
df["recipient_zip_code"] = df["recipient_zip_code"].str.replace(r"-\d{4}$", "", regex=True)

In [13]:
# Any column that starts with name_of is trimmed and uppercased
cols = [c for c in df.columns if c.startswith("name_of")]

df[cols] = df[cols].apply(lambda col: col.str.upper().str.strip())

In [14]:
# Any column that starts with product_category_ is trimmed and uppercased
cols = [c for c in df.columns if c.startswith("product_category")]

df[cols] = df[cols].apply(lambda col: col.str.upper().str.strip())

In [15]:
# Clean manufacturer name
df["applicable_manufacturer_or_applicable_gpo_making_payment_name"] = (
    df["applicable_manufacturer_or_applicable_gpo_making_payment_name"]
        .str.upper() # Uppercase
        .str.replace(r"[.,]", "", regex=True) # Remove periods
        .str.strip() # trim
)

In [16]:
# Check blank rows in manufacturer name
blank_rows = df[df["applicable_manufacturer_or_applicable_gpo_making_payment_name"]
                  .astype(str).str.strip().eq("")]
blank_rows

,change_type,covered_recipient_type,teaching_hospital_ccn,teaching_hospital_id,teaching_hospital_name,covered_recipient_profile_id,covered_recipient_npi,covered_recipient_first_name,covered_recipient_middle_name,covered_recipient_last_name,...,associated_drug_or_biological_ndc_4,associated_device_or_medical_supply_pdi_4,covered_or_noncovered_indicator_5,indicate_drug_or_biological_or_device_or_medical_supply_5,product_category_or_therapeutic_area_5,name_of_drug_or_biological_or_device_or_medical_supply_5,associated_drug_or_biological_ndc_5,associated_device_or_medical_supply_pdi_5,program_year,payment_publication_date


In [17]:
###################
# Drop blank columns
#####################
df.dropna(axis=1, how="all", inplace=True)

In [18]:
#####################
# Drop unrelated columns
#####################
drops_cols = [
    'covered_recipient_profile_id',        
    'covered_recipient_first_name', 'covered_recipient_middle_name',  # PII names unnecessary
    'covered_recipient_last_name', 'covered_recipient_name_suffix',
    'teaching_hospital_ccn', 'teaching_hospital_id', 'teaching_hospital_name', # no teaching hospitals in filtered states
#     'recipient_province',                                   # US-only dataset; always blank
    'covered_recipient_primary_type_2', 'covered_recipient_primary_type_3',  # mostly blank
#     'covered_recipient_specialty_2',                        # mostly blank
    'related_product_indicator', 'delay_in_publication_indicator',  # only Nos  zero variance
    'associated_drug_or_biological_ndc_1', 'associated_drug_or_biological_ndc_2',  # NDC not needed
    'associated_drug_or_biological_ndc_3', 'associated_drug_or_biological_ndc_4',
    'associated_drug_or_biological_ndc_5',
    'submitting_applicable_manufacturer_or_applicable_gpo_name',  # near-duplicate of payment name
    'contextual_information',                               # mostly blank
]

In [19]:
df.drop(columns = drops_cols, inplace = True) # drop blank columns

In [20]:
print(f"New Dataset Shape After Dropping Cols: {df.shape}")

New Dataset Shape After Dropping Cols: (1146735, 62)


In [21]:
df['dispute_status_for_publication'].value_counts()

dispute_status_for_publication
No     1146528
Yes        207
Name: count, dtype: int64

### Filter the General Payments Dataset

In [22]:
###################################
# filter dataset to remove Nas in total_amount_of_payment_us_dollars
###################################
df = df.dropna(subset=["total_amount_of_payment_us_dollars"])

In [23]:
###################################
# filter dataset to remove Nas in covered_recipient_npi
###################################
df = df.dropna(subset=["covered_recipient_npi"])

In [24]:
###################################
# Filter all non-physician practitioners
###################################
df = df[df["covered_recipient_type"] != "Covered Recipient Non-Physician Practitioner"]

In [25]:
### Update the wording for the nature of payments

mapping = {
    'Charitable Contribution': 'Charity',
    'Entertainment': 'Entertainment',
    'Debt forgiveness': 'Debt Forgiveness',
    'Current or prospective ownership or investment interest': 'Ownership or Investment',
    'Compensation for serving as faculty or as a speaker for a medical education program': 'Faculty or Speaker',
    'Education': 'Education',
    'Acquisitions': 'Acquisition',
    'Compensation for services other than consulting, including serving as faculty or as a speaker at a venue other than a continuing education program': 'Other Services',
    'Food and Beverage': 'Food and Beverage',
    'Consulting Fee': 'Consulting'
}

df['nature_short_descr'] = df['nature_of_payment_or_transfer_of_value'].map(mapping)


## Load specialty dimension and join onto df

In [26]:
# Load the one-row-per-(NPI, year) specialty table.
# This replaces the inline pipe-split of covered_recipient_specialty_1 and gives
# cleaner, deduplicated specialty_type / specialty_lvl1 / specialty columns
# that flow through to the providers dim and the anomaly-ready export.
specialty_npis = pd.read_csv(
    "/dsa/groups/casestudycf25/team02/silver/one_specialty_per_npi_year.csv"
)

specialty_npis = specialty_npis[
    ["rfrg_npi", "year", "specialty_type", "specialty_lvl1", "specialty"]
].copy()

# Normalise join-key dtypes to match df.
specialty_npis["rfrg_npi"] = pd.to_numeric(specialty_npis["rfrg_npi"], errors="coerce").astype("Int64")
specialty_npis["year"]     = pd.to_numeric(specialty_npis["year"],     errors="coerce").astype("Int64")

print(f"specialty_npis shape: {specialty_npis.shape}")
specialty_npis.head()


specialty_npis shape: (1265154, 5)


,rfrg_npi,year,specialty_type,specialty_lvl1,specialty
0,1043273253,2022,Medical Doctor,Allopathic & Osteopathic Physicians,Internal Medicine
1,1275631657,2022,Other,Other Service Providers,Specialist
2,1821084567,2023,Medical Doctor,Allopathic & Osteopathic Physicians,Internal Medicine
3,1821080375,2023,Medical Doctor,Allopathic & Osteopathic Physicians,Internal Medicine
4,1821219569,2023,Medical Doctor,Allopathic & Osteopathic Physicians,Obstetrics and Gynecology


In [27]:
# Merge specialty onto the main transaction frame.
# Left join preserves all transactions; unmatched NPIs receive NaN.
# validate='many_to_one' guards against fan-out if the specialty table
# ever has duplicate (NPI, year) rows.
df = df.merge(
    specialty_npis,
    left_on=["covered_recipient_npi", "program_year"],
    right_on=["rfrg_npi", "year"],
    how="left",
    validate="many_to_one",
).drop(columns=["rfrg_npi", "year"])

print(f"Shape after specialty merge: {df.shape}")
print(f"  specialty_type  NaN rate: {df['specialty_type'].isna().mean():.1%}")
print(f"  specialty_lvl1  NaN rate: {df['specialty_lvl1'].isna().mean():.1%}")
print(f"  specialty       NaN rate: {df['specialty'].isna().mean():.1%}")

# Drop the original pipe-delimited specialty_1 column  the three clean
# columns from the specialty dim supersede it.
df.drop(columns=["covered_recipient_specialty_1"], inplace=True, errors="ignore")

df[["covered_recipient_npi", "program_year", "specialty_type", "specialty_lvl1", "specialty"]].head()


Shape after specialty merge: (932908, 66)
  specialty_type  NaN rate: 0.0%
  specialty_lvl1  NaN rate: 0.0%
  specialty       NaN rate: 0.0%


,covered_recipient_npi,program_year,specialty_type,specialty_lvl1,specialty
0,1750964185,2023,Medical Doctor,Allopathic & Osteopathic Physicians,Orthopedic Surgery
1,1417944091,2023,Medical Doctor,Allopathic & Osteopathic Physicians,Orthopedic Surgery
2,1699208850,2023,Doctor of Podiatric Medicine,Podiatric Medicine & Surgery Service Providers,Podiatrist
3,1124052717,2023,Doctor of Podiatric Medicine,Podiatric Medicine & Surgery Service Providers,Podiatrist
4,1124052717,2023,Doctor of Podiatric Medicine,Podiatric Medicine & Surgery Service Providers,Podiatrist


In [28]:
df.columns

Index(['change_type', 'covered_recipient_type', 'covered_recipient_npi',
       'recipient_primary_business_street_address_line1',
       'recipient_primary_business_street_address_line2', 'recipient_city',
       'recipient_state', 'recipient_zip_code', 'recipient_country',
       'covered_recipient_primary_type_1',
       'covered_recipient_license_state_code1',
       'covered_recipient_license_state_code2',
       'covered_recipient_license_state_code3',
       'covered_recipient_license_state_code4',
       'covered_recipient_license_state_code5',
       'applicable_manufacturer_or_applicable_gpo_making_payment_id',
       'applicable_manufacturer_or_applicable_gpo_making_payment_name',
       'applicable_manufacturer_or_applicable_gpo_making_payment_state',
       'applicable_manufacturer_or_applicable_gpo_making_payment_country',
       'total_amount_of_payment_us_dollars', 'date_of_payment',
       'number_of_payments_included_in_total_amount',
       'form_of_payment_or_transf

## Add simple fraud-oriented engineered features

These features stay at the transaction grain and are intentionally simple so they
can be reused by downstream notebooks without introducing heavy modeling logic.

They are designed to surface common payment-risk patterns such as:
- unusually large average dollars per bundled payment
- delayed publication relative to payment date
- weekend or year-end timing
- travel-, consulting-, speaking-, ownership-, charity-, and third-party-related payments


In [29]:
# Create simple transaction-level engineered features for downstream fraud analysis.
# Keep the features lightweight so the silver layer stays broadly reusable.

# Protect the denominator used in average amount calculations.
payment_count = df["number_of_payments_included_in_total_amount"].fillna(1).clip(lower=1)

# Measure dollars per included payment inside a bundled record.
df["avg_amount_per_payment"] = df["total_amount_of_payment_us_dollars"] / payment_count

# Compress heavy-tailed payment amounts for downstream plots and models.
df["log_total_amount"] = np.log1p(df["total_amount_of_payment_us_dollars"])

# Convert payment timing into reusable calendar features.
df["payment_month"] = df["date_of_payment"].dt.month.astype("Int64")
df["payment_quarter"] = df["date_of_payment"].dt.quarter.astype("Int64")

# Flag weekend payments because they can be atypical for business workflows.
df["is_weekend_payment"] = df["date_of_payment"].dt.dayofweek.isin([5, 6]).astype("Int64")

# Flag year-end payments because timing pressure can create unusual patterns.
df["is_q4_payment"] = (df["payment_quarter"] == 4).astype("Int64")

# Measure the lag between the payment date and the publication date.
df["payment_to_publication_lag_days"] = (
    df["payment_publication_date"] - df["date_of_payment"]
).dt.days.astype("Int64")

# Normalize the payment nature text for rule-style flags.
nature_text = df["nature_short_descr"].fillna(df["nature_of_payment_or_transfer_of_value"]).fillna("").str.lower()

# Flag travel-related transfers of value.
df["is_travel_payment"] = (
    nature_text.str.contains("travel|lodging", regex=True)
    | df["city_of_travel"].notna()
    | df["state_of_travel"].notna()
    | df["country_of_travel"].notna()
).astype("Int64")

# Flag consulting arrangements that can overlap with kickback investigations.
df["is_consulting_payment"] = nature_text.str.contains("consult", regex=True).astype("Int64")

# Flag speaker or education-style compensation categories.
df["is_speaker_payment"] = nature_text.str.contains("speaker|faculty|education", regex=True).astype("Int64")

# Flag ownership-linked records using the payment nature or ownership indicator.
df["is_ownership_payment"] = (
    nature_text.str.contains("ownership|investment", regex=True)
    | df["physician_ownership_indicator"].fillna("").str.lower().eq("yes")
).astype("Int64")

# Flag charitable transfers because they can behave differently from direct payments.
df["is_charity_payment"] = (
    nature_text.str.contains("charit", regex=True)
    | df["charity_indicator"].fillna("").str.lower().eq("yes")
).astype("Int64")

# Flag third-party payment structures because they can obscure the ultimate flow of value.
df["is_third_party_payment"] = (
    df["third_party_payment_recipient_indicator"].fillna("").str.lower().ne("")
    | df["third_party_equals_covered_recipient_indicator"].fillna("").str.lower().eq("yes")
    | df["name_of_third_party_entity_receiving_payment_or_transfer_of_value"].notna()
).astype("Int64")

# Review the added feature set.
engineered_cols = [
    "avg_amount_per_payment",
    "log_total_amount",
    "payment_month",
    "payment_quarter",
    "is_weekend_payment",
    "is_q4_payment",
    "payment_to_publication_lag_days",
    "is_travel_payment",
    "is_consulting_payment",
    "is_speaker_payment",
    "is_ownership_payment",
    "is_charity_payment",
    "is_third_party_payment",
]

df[engineered_cols].head()


,avg_amount_per_payment,log_total_amount,payment_month,payment_quarter,is_weekend_payment,is_q4_payment,payment_to_publication_lag_days,is_travel_payment,is_consulting_payment,is_speaker_payment,is_ownership_payment,is_charity_payment,is_third_party_payment
0,10.58,2.449279,9,3,0,0,640,0,0,0,0,0,1
1,8.23,2.222459,6,2,0,0,741,0,0,0,0,0,1
2,10.33,2.427454,2,1,0,0,880,0,0,0,0,0,1
3,6.57,2.024193,6,2,0,0,733,1,0,0,0,0,1
4,90.79,4.519503,7,3,0,0,714,0,0,0,0,0,1


In [30]:
print("Check if there are any duplicate record ids in final dataset: ")
df["record_id"].duplicated().any()

Check if there are any duplicate record ids in final dataset: 


False

In [31]:
print(f"New Dataset Shape After Dropping Rows: {df.shape}")

New Dataset Shape After Dropping Rows: (932908, 78)


## CREATE DIMENSIONS TO REDUCE DATA LOAD

Split the general payments data into domain specific dimensions. Including manufacturing, provider info, products info, and then the general fact records table. 

########################################
# CREATE PROVIDERS DIM
########################################

In [32]:

df.columns

Index(['change_type', 'covered_recipient_type', 'covered_recipient_npi',
       'recipient_primary_business_street_address_line1',
       'recipient_primary_business_street_address_line2', 'recipient_city',
       'recipient_state', 'recipient_zip_code', 'recipient_country',
       'covered_recipient_primary_type_1',
       'covered_recipient_license_state_code1',
       'covered_recipient_license_state_code2',
       'covered_recipient_license_state_code3',
       'covered_recipient_license_state_code4',
       'covered_recipient_license_state_code5',
       'applicable_manufacturer_or_applicable_gpo_making_payment_id',
       'applicable_manufacturer_or_applicable_gpo_making_payment_name',
       'applicable_manufacturer_or_applicable_gpo_making_payment_state',
       'applicable_manufacturer_or_applicable_gpo_making_payment_country',
       'total_amount_of_payment_us_dollars', 'date_of_payment',
       'number_of_payments_included_in_total_amount',
       'form_of_payment_or_transf

In [33]:
providers = df[[
    "covered_recipient_npi",
    "covered_recipient_type",
    "covered_recipient_primary_type_1",
    "specialty_type",
    "specialty_lvl1",
    "specialty",
    "covered_recipient_license_state_code1",
    "covered_recipient_license_state_code2",
    "covered_recipient_license_state_code3",
    "covered_recipient_license_state_code4",
    "covered_recipient_license_state_code5",
    "recipient_primary_business_street_address_line1",
    "recipient_primary_business_street_address_line2",
    "recipient_city",
    "recipient_state",
    "recipient_zip_code",
    "recipient_country",
    "program_year",
]]


In [34]:
providers.sort_values(by = "program_year", ascending = False)

,covered_recipient_npi,covered_recipient_type,covered_recipient_primary_type_1,specialty_type,specialty_lvl1,specialty,covered_recipient_license_state_code1,covered_recipient_license_state_code2,covered_recipient_license_state_code3,covered_recipient_license_state_code4,covered_recipient_license_state_code5,recipient_primary_business_street_address_line1,recipient_primary_business_street_address_line2,recipient_city,recipient_state,recipient_zip_code,recipient_country,program_year
0,1750964185,Covered Recipient Physician,Medical Doctor,Medical Doctor,Allopathic & Osteopathic Physicians,Orthopedic Surgery,IL,NaN,NaN,NaN,NaN,600 S PAULINA ST,NaN,CHICAGO,IL,60612,United States,2023
564821,1285617548,Covered Recipient Physician,Doctor of Dentistry,Doctor of Dentistry,Dental Providers,Dentist,TN,NaN,NaN,NaN,NaN,776 S MAIN ST,NaN,ASHLAND CITY,TN,37015,United States,2023
564823,1295727154,Covered Recipient Physician,Medical Doctor,Medical Doctor,Allopathic & Osteopathic Physicians,Orthopedic Surgery,AR,NaN,NaN,NaN,NaN,10301 KANIS RD,NaN,LITTLE ROCK,AR,72205,United States,2023
564822,1730134065,Covered Recipient Physician,Medical Doctor,Medical Doctor,Allopathic & Osteopathic Physicians,Radiology,AR,NaN,NaN,NaN,NaN,9500 KANIS RD STE 330,NaN,LITTLE ROCK,AR,72205,United States,2023
714671,1558362822,Covered Recipient Physician,Medical Doctor,Medical Doctor,Allopathic & Osteopathic Physicians,General Practice,MO,NaN,NaN,NaN,NaN,408 S BROADVIEW ST,NaN,CAPE GIRARDEAU,MO,63703,United States,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235479,1932596681,Covered Recipient Physician,Medical Doctor,Medical Doctor,Allopathic & Osteopathic Physicians,Obstetrics and Gynecology,OK,NaN,NaN,NaN,NaN,4502 E 41ST ST RM 2H08,NaN,TULSA,OK,74135,United States,2021
235478,1669487054,Covered Recipient Physician,Doctor of Osteopathy,Medical Doctor,Allopathic & Osteopathic Physicians,Obstetrics and Gynecology,MO,NaN,NaN,NaN,NaN,3201 MCINTOSH CIR,NaN,JOPLIN,MO,64804,United States,2021
235477,1760460208,Covered Recipient Physician,Medical Doctor,Medical Doctor,Allopathic & Osteopathic Physicians,Urology,TN,NaN,NaN,NaN,NaN,107 GLEN OAK BLVD STE 100,NaN,HENDERSONVILLE,TN,37075,United States,2021
235476,1811378219,Covered Recipient Physician,Medical Doctor,Medical Doctor,Allopathic & Osteopathic Physicians,Urology,FL,NaN,NaN,NaN,NaN,1924 ALCOA HWY,NaN,KNOXVILLE,TN,37920,United States,2021


In [35]:
# KEEP ONLY MOST RECENT PROVIDER INFORMATION
providers = providers.drop_duplicates(keep='first').copy()

In [36]:
providers.to_csv("/dsa/groups/casestudycf25/team02/silver/general_payments_providers_clean.csv", index =False)

In [37]:
del providers

########################################
# CREATE PRODUCTS DIM
########################################

In [38]:
#  1 to 5 cols
slots = range(1, 6)

long_frames = []

for i in slots:
    tmp = df[[
        'record_id',
        f'associated_device_or_medical_supply_pdi_{i}',
        f'name_of_drug_or_biological_or_device_or_medical_supply_{i}',
        f'covered_or_noncovered_indicator_{i}',
        f'product_category_or_therapeutic_area_{i}',
        f'indicate_drug_or_biological_or_device_or_medical_supply_{i}'
    ]].copy()

    tmp.columns = [
        'record_id',
        'pdi',
        'product_name',
        'covered_indicator',
        'product_category',
        'drug_or_device_indicator'
    ]

    # Keep the source slot for debugging or tracking
    tmp['slot'] = i

    long_frames.append(tmp)

# Concatenate into single long table
products = pd.concat(long_frames, ignore_index=True)

# Drop empty rows where no product was specified
products = products.dropna(subset=['pdi', 'product_name'], how='all')

products

,record_id,pdi,product_name,covered_indicator,product_category,drug_or_device_indicator,slot
0,1006679101,07613327131796,PRO,Covered,TRAUMA & EXTREMITIES,Device,1
1,1006696505,NaN,ACTISHIELD,Covered,TRAUMA & EXTREMITIES,Device,1
2,1006744861,00840420115799,VALOR,Covered,TRAUMA & EXTREMITIES,Device,1
3,1006725903,00889797071963,PROSTEP,Covered,TRAUMA & EXTREMITIES,Device,1
4,1006725915,07613154969463,VARIAX,Covered,TRAUMA & EXTREMETIES,Device,1
...,...,...,...,...,...,...,...
4650777,947088335,10381780113744,PRIMATRIX,Covered,RECON-SKIN AND WOUND,Device,5
4660039,1007012903,04546540375162,GAMMA,Covered,TRAUMA & EXTREMETIES,Device,5
4660118,1007122623,04546540375162,GAMMA,Covered,TRAUMA & EXTREMETIES,Device,5
4660144,1007103379,04546540086150,ASNIS,Covered,TRAUMA & EXTREMETIES,Device,5


In [39]:
# get a df of product names and pdis: looks like [id, [name1,name2,3]]
names_list_product_ids = (
    products.groupby('pdi')['product_name']
           .apply(lambda s: s.dropna().unique())
           .to_dict()
)

# map names to a list of ids and only apply it if there is a single valid id
single_pdi_map = {
    name: vals[0]
    for name, vals in names_list_product_ids.items()
    if len(vals) == 1
}

# Create new temp column 
products['name_filled'] = products['pdi'].map(single_pdi_map)

In [40]:
# if pdi is null then fill with the new pdi_filled colum
products['product_name'] = products['product_name'].fillna(products['name_filled'])

In [41]:
# drop the temp column
products = products.drop(columns=['name_filled'])

In [42]:
########################################
# CREATE PRODUCT ROLLUP AT RECORD LEVEL
########################################
# Create product_indicator column combining covered status and product type
# This rollup will be attached to the main df so downstream scripts can use it directly

products["product_indicator"] = (
    products["covered_indicator"].str.lower().fillna("unknown") + "_" + 
    products["drug_or_device_indicator"].str.lower().fillna("unknown")
)

# Pivot to get counts of each product type per record_id
product_pivot = (
    pd.crosstab(products['record_id'], products['product_indicator'])
    .reset_index()
)

# Convert counts to binary indicators (1 if any product of that type, 0 otherwise)
# Keep as counts for more granular downstream analysis
print(f"Product pivot shape: {product_pivot.shape}")
print(f"Product indicator columns: {[c for c in product_pivot.columns if c != 'record_id']}")


Product pivot shape: (919536, 9)
Product indicator columns: ['covered_biological', 'covered_device', 'covered_drug', 'covered_medical supply', 'non-covered_device', 'non-covered_drug', 'non-covered_medical supply', 'non-covered_unknown']


In [43]:
########################################
# MERGE PRODUCT ROLLUP BACK TO MAIN DF
########################################
# Left join to preserve all records, filling missing with 0

df = df.merge(
    product_pivot,
    on='record_id',
    how='left',
    validate='one_to_one'
)

# Fill any NaN product columns with 0 (records with no associated products)
product_cols = [c for c in product_pivot.columns if c != 'record_id']
df[product_cols] = df[product_cols].fillna(0).astype(int)

print(f"Added product rollup columns to df: {product_cols}")
print(f"Updated df shape: {df.shape}")

# Clean up the pivot table
del product_pivot


Added product rollup columns to df: ['covered_biological', 'covered_device', 'covered_drug', 'covered_medical supply', 'non-covered_device', 'non-covered_drug', 'non-covered_medical supply', 'non-covered_unknown']
Updated df shape: (932908, 86)


In [44]:
products.shape

(1138868, 8)

In [45]:
products[
#     products['pdi'].isna() &
    products['product_name'].isna()
]

,record_id,pdi,product_name,covered_indicator,product_category,drug_or_device_indicator,slot,product_indicator
28,1006853491,04546540355355,NaN,Covered,CRANIOMAXILLOFACIAL,Device,1,covered_device
183,1016917215,07613327503265,NaN,Covered,NAVIGATION,Device,1,covered_device
4447,1061049737,10886982231215,NaN,Covered,TRAUMA,Device,1,covered_device
4494,1061049931,10886982231215,NaN,Covered,TRAUMA,Device,1,covered_device
4558,1061186135,07613327375794,NaN,Covered,REPROCESSED SURGICAL,Device,1,covered_device
...,...,...,...,...,...,...,...,...
4432646,806699795,07613327370713,NaN,Covered,CRANIOMAXILLOFACIAL,Device,5,covered_device
4445412,922180859,07613327370713,NaN,Covered,CRANIOMAXILLOFACIAL,Device,5,covered_device
4452962,922797685,07613154185702,NaN,Covered,CRANIOMAXILLOFACIAL,Device,5,covered_device
4506747,806601753,07613327291315,NaN,Covered,CRANIOMAXILLOFACIAL,Device,5,covered_device


In [46]:
# Every column can still be nullable except for record id
products.to_csv("/dsa/groups/casestudycf25/team02/silver/general_payments_products_clean.csv", index =False)

In [47]:
del products

########################################
# CREATE Manufacturers DIM
########################################

In [48]:

manufacturers = df[[
"applicable_manufacturer_or_applicable_gpo_making_payment_id",
"applicable_manufacturer_or_applicable_gpo_making_payment_name",
"applicable_manufacturer_or_applicable_gpo_making_payment_state",
"applicable_manufacturer_or_applicable_gpo_making_payment_country", "program_year"]]

In [49]:
manufacturers.sort_values(by = "program_year", ascending = False)

,applicable_manufacturer_or_applicable_gpo_making_payment_id,applicable_manufacturer_or_applicable_gpo_making_payment_name,applicable_manufacturer_or_applicable_gpo_making_payment_state,applicable_manufacturer_or_applicable_gpo_making_payment_country,program_year
0,100000010503,STRYKER CORPORATION,MI,United States,2023
564821,100000005497,ULTRADENT PRODUCTS INC,UT,United States,2023
564823,100000010503,STRYKER CORPORATION,MI,United States,2023
564822,100000010503,STRYKER CORPORATION,MI,United States,2023
714671,100000005734,COCHLEAR AMERICAS,CO,United States,2023
...,...,...,...,...,...
235479,100000806850,AXONICS INC,CA,United States,2021
235478,100000806850,AXONICS INC,CA,United States,2021
235477,100000806850,AXONICS INC,CA,United States,2021
235476,100000806850,AXONICS INC,CA,United States,2021


In [50]:
manufacturers = df[[
"applicable_manufacturer_or_applicable_gpo_making_payment_id",
"applicable_manufacturer_or_applicable_gpo_making_payment_name",
"applicable_manufacturer_or_applicable_gpo_making_payment_state",
"applicable_manufacturer_or_applicable_gpo_making_payment_country"]]

In [51]:
manufacturers = manufacturers.drop_duplicates(keep='first').copy()

In [52]:
# Drop duplicates based on manufacturing id
manufacturers = manufacturers.drop_duplicates(keep='first',
                                              subset= "applicable_manufacturer_or_applicable_gpo_making_payment_id").copy()

In [53]:
print("Check if there are any duplicate record ids in final dataset: ")
manufacturers["applicable_manufacturer_or_applicable_gpo_making_payment_id"].duplicated().any()

Check if there are any duplicate record ids in final dataset: 


False

In [54]:
manufacturers.to_csv("/dsa/groups/casestudycf25/team02/silver/general_payments_manufacturers_clean.csv", index =False)

In [55]:
del manufacturers

########################################
# CREATE REMAINING FACT TAB
########################################

This keeps the transaction-level General Payments fact table that feeds the next
pipeline step. The LEIE/OIG target is attached first; then the sliced export
carries that transaction-level `target` forward into `open_payments_silver`.


In [56]:
col_totals = df.select_dtypes(include="number").sum()
print("All Column Totals")
print("-" *50)
print(col_totals.apply(lambda x: f"{x:.0f}"))

All Column Totals
--------------------------------------------------
covered_recipient_npi                          1401265398803749
total_amount_of_payment_us_dollars                    507462114
number_of_payments_included_in_total_amount              946273
program_year                                         1886444731
avg_amount_per_payment                                500866913
log_total_amount                                        3445585
payment_month                                           6256551
payment_quarter                                         2386706
is_weekend_payment                                        65531
is_q4_payment                                            243977
payment_to_publication_lag_days                       977285241
is_travel_payment                                         83871
is_consulting_payment                                     25314
is_speaker_payment                                        15869
is_ownership_payment               

## LEIE List

In [57]:
# Bring in the LEIE list
leie = pd.read_csv("/dsa/groups/casestudycf25/team02/silver/leie_with_valid_npi_clean.csv")

leie.shape

(1884, 18)

In [58]:
leie["npi"] = pd.to_numeric(leie["npi"], errors = "coerce").astype("Int64")

In [59]:
leie["excldate"] = pd.to_datetime(leie["excldate"])      # parse as datetime (if not already)
leie["year_leie"] = leie["excldate"].dt.year

In [60]:
leie.head(1)

,general,specialty,npi,excltype,excldate,num_exclusions_alltime,num_exclusion_types_alltime,list_exclusion_types_alltime,num_addresses_alltime,new_id,fraud_flag,excl_1128a1,excl_1128a2,excl_1128a3,excl_1128b5,excl_1128b6,excl_1128b7,excl_brch cia,year_leie
0,IND- LIC HC SERV PRO,DENTIST,1861529091,1128a3,2024-03-20,1.0,1.0,['1128a3'],1.0,AAMIR-WAHAB-nan-1979-11-16,1,0,0,1,0,0,0,0,2024


In [61]:
leie =  leie[["npi","excldate", "fraud_flag"]].copy()

### Attach LEIE-derived target labels at the transaction grain

Because `open_payments_silver` runs next in the pipeline, this notebook is the
right place to do the **date-aware LEIE/OIG merge once** at the transaction
level. The downstream notebook then rolls that existing transaction-level
`target` up to the provider-year grain; it should not repeat the LEIE merge.


In [62]:
# Transaction-level LEIE label attachment for General Payments
# Keep the merge at the transaction grain; downstream steps inherit this label.
txns_leie = df.merge(
    leie,
    left_on="covered_recipient_npi",
    right_on="npi",
    how="left",
    validate="many_to_one"
)

txn_cond = txns_leie["date_of_payment"] <= txns_leie["excldate"]

txn_leie_cols = [
    "npi",
    "excldate",
    "fraud_flag"
]

# Blank out LEIE-derived values whenever the exclusion was not yet active.
txns_leie.loc[~txn_cond, txn_leie_cols] = np.nan

# Binary transaction-level modeling label.
txns_leie["target"] = txns_leie["fraud_flag"].fillna(0).astype("Int64")

txns_leie.drop(columns=["fraud_flag"], inplace=True)

df = txns_leie.copy()

print("Transaction-level dataset with target:", df.shape)
df[["covered_recipient_npi", "date_of_payment", "excldate", "target"]].head()


Transaction-level dataset with target: (932908, 89)


,covered_recipient_npi,date_of_payment,excldate,target
0,1750964185,2023-09-29,NaT,0
1,1417944091,2023-06-20,NaT,0
2,1699208850,2023-02-01,NaT,0
3,1124052717,2023-06-28,NaT,0
4,1124052717,2023-07-17,NaT,0


In [63]:
df['target'].value_counts()

target
0    932441
1       467
Name: count, dtype: Int64

## DME & Medical Supply Equipment Categorization

This section scans product names and therapeutic areas against DME keywords
that Missouri investigators prioritize for fraud detection, plus all 12 HCPCS
DME category groupings.

The resulting columns (`dme_categories`, `dme_any`, `dme_count`) are added at
the record level and will flow into the sliced export for downstream analysis.


In [64]:
# =============================================================================
# DME & MEDICAL SUPPLY EQUIPMENT ANALYSIS 
# Covers the Medicaid DME/supply categories Missouri investigators
# prioritize for fraud, plus all 12 HCPCS DME category groupings.
# =============================================================================

DME_KEYWORDS = {

    # ── Original MO high-risk categories (corrected) ────────────────────

    'Glucose Monitors':     ['GLUCOSE', 'FREESTYLE', 'LIBRE', 'DEXCOM', 'OMNIPOD',
                             'GLUCOMETER', 'BLOOD SUGAR', 'CGM',
                             'CONTINUOUS GLUCOSE', 'POGO', 'INSULINX'],

    'Urological Supplies':  ['UROLOGY', 'UROLOGICAL', 'URINARY',
                             'AXONICS', 'UROLIFT',
                             'BLADDER', 'CONTINENCE CARE',
                             'INCONTINENCE', 'FOLEY',
                             'PELVIC HEALTH', 'GENITOURINARY',
                             'ENDOUROLOGY', 'CATHETER CARE'],

    'Surgical Dressings':   ['DRESSING', 'WOUND CARE', 'BANDAGE', 'GAUZE',
                             'TEGADERM', 'SURGICAL DRESS', 'BIATAIN',
                             'PURACOL', 'IOPLEX', 'WOUND MATRIX',
                             'NEGATIVE PRESSURE SYSTEMS', 'NPWT',
                             'FOAM DRESSING'],

    'Positive Airway':      ['CPAP', 'BIPAP', 'BILEVEL', 'SLEEP APNEA', 'PAP ',
                             'RESMED', 'RESPIRONICS', 'DREAMSTATION',
                             'CPAP/BILEVEL'],

    'Lower Limb Orthotics': ['ORTHOTIC', 'ORTHOSES',
                             'ORTHOSES: PREFAB',
                             'DYNASPLINT',
                             'LOWER LIMB', 'ANKLE BRACE', 'KNEE BRACE',
                             'ANKLE-FOOT', 'FOOT ORTHOT',
                             'CRANIAL ORTHOTIC'],

    'Parenteral Nutrition': ['PARENTERAL', 'TPN', 'ENTERAL',
                             'FEEDING', 'TUBE FEED', 'ENTERAL FEEDING',
                             'IPS - NUTRITION'],

    'Ventilators':          ['VENTILATOR', 'RESPIRATOR',
                             'TRACHEOSTOMY',
                             'BREATHING MACHINE', 'VENTILATION SYSTEM',
                             'VENTILATOR SYSTEM', 'VENTILATORY'],

    # ── HCPCS: Other Supplies Incl Diabetes Supplies & Contraceptives ──

    'Diabetes Supplies & Contraceptives':
                            ['DIABETES', 'DIABETIC', 'DIABETES CARE',
                             'DIABETES TESTING', 'LANCET', 'TEST STRIP',
                             'PEN NEEDLE', 'INSULIN DELIVER', 'INSULIN SYRINGE',
                             'INPEN', 'V-GO',
                             'CONTRACEPTI', 'CONDOM', 'DIAPHRAGM'],

    # ── HCPCS: Accessories for Oxygen Delivery Devices ─────────────────

    'Oxygen Accessories':   ['NASAL CANNULA', 'OXYGEN MASK', 'O2 MASK',
                             'OXYGEN TUBING', 'O2 TUBING',
                             'OXYGEN ACCESS', 'OXYGEN REGULAT', 'O2 REGULAT',
                             'OPTIFLOW', 'BREVIDA',
                             '/ESON',
                             'VITERA', 'SIMPLUS', 'EVORA'],

    # ── HCPCS: Oxygen Delivery Systems and Related Supplies ────────────

    'Oxygen Equipment':     ['OXYGEN', 'CONCENTRATOR',
                             'OXYGEN CONCENTRAT',
                             'LIQUID OXYGEN', 'PORTABLE OXYGEN',
                             'INOGEN', 'TOPICAL WOUND OXYGEN',
                             'OXYGEN CHAMBER', 'RESPIRATORY CARE',
                             'AIRVO'],

    # ── HCPCS: Humidifiers and Nebulizers with Related Equipment ───────

    'Humidifiers & Nebulizers':
                            ['NEBULIZ', 'NEBULISER', 'HUMIDIF', 'AEROSOL',
                             'COMPRESSOR NEBUL', 'AEROGEN',
                             'NEBULIZER SYSTEM', 'NEBULIZER SOLUTION',
                             'MIST THERAPY'],

    # ── HCPCS: Wheelchairs, Components, and Accessories ────────────────

    'Wheelchair Access':    ['WHEELCHAIR', 'WHEEL CHAIR',
                             'SCOOTER', 'POWER CHAIR',
                             'SEATING SYSTEM', 'WHEELCHAIR CUSHION',
                             'JOYSTICK', 'TILT-IN-SPACE',
                             'PERMOBIL', 'QUICKIE', 'TILITE',
                             'JAZZY', 'ROHO'],

    # ── HCPCS: Pharmacy Supply and Dispensing Fees ─────────────────────

    'Pharmacy Dispensing':  ['PHARMACY', 'DISPENSING FEE',
                             'PHARMACY SUPPLY', 'PHARMACY PRODUCTS',
                             'INTEGRATED PHARMACY', 'DISPENS',
                             'PYXIS', 'OMNICELL'],

    # ── HCPCS: Infusion Pumps and Supplies ─────────────────────────────

    'Infusion Pumps':       ['INFUSION', 'IV PUMP', 'INSULIN PUMP',
                             'T-SLIM', 'T:SLIM',
                             'MEDTRONIC PUMP',
                             'INFUSION SYSTEM', 'DRUG INFUSION',
                             'PUMP, INFUSION', 'DURABLE PUMP',
                             'TRINAV'],

    # ── HCPCS: Inhalation Solutions ────────────────────────────────────

    'Inhalation Solutions': ['INHALATION', 'ALBUTEROL', 'IPRATROPIUM',
                             'BUDESONIDE', 'LEVALBUTEROL', 'ARFORMOTEROL',
                             'CROMOLYN', 'ACETYLCYSTEINE', 'XOPENEX',
                             'PROVENTIL', 'PERFOROMIST', 'BROVANA',
                             'DUONEB', 'COMBIVENT', 'PULMICORT',
                             'INHALATION SOLUTION', 'INHALER'],

    # ── HCPCS: Breathing Aids ──────────────────────────────────────────

    'Breathing Aids':       ['AIRWAY',
                             'INSPIRE UPPER AIRWAY',
                             'SYNCLARA', 'COUGH SYSTEM', 'VEST SYSTEM',
                             'VOLARA', 'CHEST WALL OSCILLAT',
                             'SPIROMETER',
                             'BREATHING TUBE'],

    # ── HCPCS: Hospital Beds and Associated Supplies ───────────────────

    'Hospital Beds':        ['HOSPITAL BED', 'BED RAIL', 'BED SIDE RAIL',
                             'MATTRESS OVERLAY', 'ALTERNATING PRESSURE',
                             'LOW AIR LOSS', 'TRAPEZE', 'OVERBED TABLE',
                             'SEMI-ELECTRIC BED', 'FULL-ELECTRIC BED',
                             'BED PAN', 'BED PAD', 'PRESSURE REDUC'],

    # ── HCPCS: Replacement Batteries ───────────────────────────────────

    'Replacement Batteries':['BATTERY', 'BATTERIES', 'LITHIUM ION',
                             'BATTERY CHARGER', 'BATTERY PACK',
                             'BATTERY POWER', 'RECHARG', 'REPLACE BATTER'],

    # ── HCPCS: Various Medical Supplies Incl Tapes & Surg Dressings ───

    'Tapes & Medical Supplies':
                            ['ADHESIVE TAPE', 'SURGICAL TAPE ',
                             'ELASTIC BANDAGE', 'COMPRESSION BANDAGE',
                             'STOCKINETTE', 'CAST SUPPLIES',
                             'TAPES AND WRAPS', 'MEDICAL SUPPLIES',
                             'PERMATAPE', 'DYNATAPE'],
}


In [65]:
def match_dme_category(row):
    """Check product name AND therapeutic area against DME keywords."""
    text_fields = []
    # Check all 5 product name slots
    for i in range(1, 6):
        col = f'name_of_drug_or_biological_or_device_or_medical_supply_{i}'
        if col in row.index and pd.notna(row[col]):
            text_fields.append(str(row[col]).upper())
    # Also check therapeutic area / product category
    for i in range(1, 6):
        col = f'product_category_or_therapeutic_area_{i}'
        if col in row.index and pd.notna(row[col]):
            text_fields.append(str(row[col]).upper())

    combined = ' '.join(text_fields)
    matched = []
    for category, keywords in DME_KEYWORDS.items():
        for kw in keywords:
            if kw in combined:
                matched.append(category)
                break
    return matched


print("Scanning all records for DME/supply equipment matches...")
df['dme_categories'] = df.apply(match_dme_category, axis=1)
df['dme_any'] = df['dme_categories'].apply(len) > 0
df['dme_count'] = df['dme_categories'].apply(len)

print(f"DME/supply-related records: {df['dme_any'].sum():,} ({df['dme_any'].mean()*100:.1f}% of all)")


Scanning all records for DME/supply equipment matches...
DME/supply-related records: 108,344 (11.6% of all)


In [66]:
# Show breakdown by DME category
from collections import Counter

dme_df = df[df['dme_any']].copy()
cat_counts = Counter()
for cats in dme_df['dme_categories']:
    for c in cats:
        cat_counts[c] += 1

print(f"\n{'Category':<40} {'Records':>8}")
print("-" * 50)
for cat, cnt in cat_counts.most_common():
    print(f"  {cat:<38} {cnt:>6}")

del dme_df  # Clean up temp df



Category                                  Records
--------------------------------------------------
  Glucose Monitors                        44668
  Diabetes Supplies & Contraceptives      28826
  Urological Supplies                     26930
  Surgical Dressings                      22598
  Infusion Pumps                           8654
  Lower Limb Orthotics                     2755
  Breathing Aids                           2285
  Ventilators                              2205
  Oxygen Equipment                         1759
  Positive Airway                           902
  Parenteral Nutrition                      102
  Tapes & Medical Supplies                   93
  Humidifiers & Nebulizers                   55
  Oxygen Accessories                         43
  Pharmacy Dispensing                        19
  Replacement Batteries                      17


In [71]:
# Create the sliced transaction table after `target` exists so the next pipeline
# step can use the upstream transaction-level label directly.
# UPDATED: Now includes product rollup columns AND DME category columns for downstream use

# Get the product indicator columns that were added
product_indicator_cols = [c for c in df.columns if c.startswith('covered_') or c.startswith('non-covered_')]
product_indicator_cols = [c for c in product_indicator_cols if c not in [
    'covered_recipient_type', 'covered_recipient_npi', 
    'covered_recipient_primary_type_1', 'covered_recipient_license_state_code1',
    'covered_recipient_license_state_code2', 'covered_recipient_license_state_code3',
    'covered_recipient_license_state_code4', 'covered_recipient_license_state_code5',
    'covered_or_noncovered_indicator_1', 'covered_or_noncovered_indicator_2',
    'covered_or_noncovered_indicator_3', 'covered_or_noncovered_indicator_4',
    'covered_or_noncovered_indicator_5'
]]

print(f"Product indicator columns to include: {product_indicator_cols}")

# DME category columns
dme_cols = ['dme_categories', 'dme_any', 'dme_count']
# DME binary columns (for rollup in open_payments_silver)
DME_BINARY_COLS = [
    'dme_glucose_monitors',
    'dme_urological_supplies',
    'dme_surgical_dressings',
    'dme_positive_airway',
    'dme_lower_limb_orthotics',
    'dme_parenteral_nutrition',
    'dme_ventilators',
    'dme_diabetes_supplies_and_contraceptives',
    'dme_oxygen_accessories',
    'dme_oxygen_equipment',
    'dme_humidifiers_and_nebulizers',
    'dme_wheelchair_access',
    'dme_pharmacy_dispensing',
    'dme_infusion_pumps',
    'dme_inhalation_solutions',
    'dme_breathing_aids',
    'dme_hospital_beds',
    'dme_replacement_batteries',
    'dme_tapes_and_medical_supplies',
]
print(f"DME columns to include: {dme_cols}")

base_cols = [
    'record_id', 'payment_publication_date', 'date_of_payment', 'program_year',
    'covered_recipient_npi',
    'applicable_manufacturer_or_applicable_gpo_making_payment_id',
    'total_amount_of_payment_us_dollars',
    'number_of_payments_included_in_total_amount',
    'form_of_payment_or_transfer_of_value',
    'nature_of_payment_or_transfer_of_value', 'nature_short_descr',
    'city_of_travel',
    'state_of_travel', 'country_of_travel',
    'physician_ownership_indicator',
    'third_party_payment_recipient_indicator',
    'name_of_third_party_entity_receiving_payment_or_transfer_of_value',
    'charity_indicator', 'third_party_equals_covered_recipient_indicator',
    'dispute_status_for_publication',
    'avg_amount_per_payment',
    'log_total_amount',
    'payment_month',
    'payment_quarter',
    'is_weekend_payment',
    'is_q4_payment',
    'payment_to_publication_lag_days',
    'is_travel_payment',
    'is_consulting_payment',
    'is_speaker_payment',
    'is_ownership_payment',
    'is_charity_payment',
    'is_third_party_payment',
    'target'
]

# Combine base columns with product indicator columns and DME columns
sliced_cols = base_cols + product_indicator_cols + dme_cols #+ DME_BINARY_COLS

sliced_df = df[sliced_cols].copy()

sliced_df.to_csv("/dsa/groups/casestudycf25/team02/silver/general_payments-sliced_clean.csv", index=False)

print(f"Sliced General Payments export shape: {sliced_df.shape}")
print(f"Product columns included: {product_indicator_cols}")
print(f"DME columns included: {dme_cols}")


Product indicator columns to include: ['covered_biological', 'covered_device', 'covered_drug', 'covered_medical supply', 'non-covered_device', 'non-covered_drug', 'non-covered_medical supply', 'non-covered_unknown']
DME columns to include: ['dme_categories', 'dme_any', 'dme_count']
Sliced General Payments export shape: (932908, 45)
Product columns included: ['covered_biological', 'covered_device', 'covered_drug', 'covered_medical supply', 'non-covered_device', 'non-covered_drug', 'non-covered_medical supply', 'non-covered_unknown']
DME columns included: ['dme_categories', 'dme_any', 'dme_count']


##############################################################
# Return the intact and large dataset as zipped CSV in chunks
##############################################################

This dataset is before the dimensions splitting but should be cleaned the same way.

## Prepare `cms_general_payments.csv` for Anomaly Detection (ECOD, INNE, Isolation Forest)

All three algorithms require a **fully numeric, finite-valued feature matrix** with no NaNs or Infs.
The specialty dimension (`specialty_type`, `specialty_lvl1`, `specialty`) was joined onto `df` earlier in
the notebook (replacing the inline pipe-split), so it is available directly  no secondary merge is needed here.

This section:
1. Selects numeric and categorical columns that carry payment-risk signal.
2. Label-encodes categorical columns (dense encoding keeps the matrix compact for ECOD / INNE).
3. Coerces all features to `float64`.
4. Replaces ±Inf values with NaN, then median-imputes any remaining NaNs.
5. Winsorizes continuous features at the 1st/99th percentile to prevent extreme outliers from dominating ECOD's empirical CDF and INNE's nearest-neighbour distances.
6. Writes a standalone `cms_general_payments_anomaly_ready.csv`.

**Feature groups included:**

| Group | Columns | Notes |
|---|---|---|
| Continuous (winsorized) | `total_amount_of_payment_us_dollars`, `avg_amount_per_payment`, `log_total_amount`, `number_of_payments_included_in_total_amount`, `payment_to_publication_lag_days`, `payment_month`, `payment_quarter`, `program_year` | Core payment magnitude & timing |
| Binary flags | `is_weekend_payment`, `is_q4_payment`, `is_travel_payment`, `is_consulting_payment`, `is_speaker_payment`, `is_ownership_payment`, `is_charity_payment`, `is_third_party_payment` | Payment-type risk patterns |
| Categorical (label-encoded) | `nature_short_descr`, `form_of_payment_or_transfer_of_value`, `covered_recipient_primary_type_1`, `recipient_state`, `specialty_type`, `specialty_lvl1`, `specialty` | Contextual risk  specialty cols flow from the upstream join |

> **Why label encoding over one-hot?**  
> ECOD, INNE, and Isolation Forest all work on the raw numeric matrix.  
> One-hot encoding expands high-cardinality columns (e.g. `specialty`) into many sparse columns and causes the curse of dimensionality to degrade INNE's nearest-neighbour density estimates.  
> Label encoding preserves ordinality implicitly accepted by tree-based Isolation Forest and keeps the matrix dense.


In [72]:
########################################
# CREATE BINARY DME CATEGORY COLUMNS
########################################
# Explode dme_categories list into individual binary columns for anomaly detection
# Each DME category becomes a 0/1 column (e.g., dme_glucose_monitors, dme_surgical_dressings)

# Get all unique DME category names from the keywords dict
all_dme_categories = list(DME_KEYWORDS.keys())

# Create binary column for each DME category
for cat in all_dme_categories:
    col_name = 'dme_' + cat.lower().replace(' ', '_').replace('&', 'and').replace('-', '_')
    df[col_name] = df['dme_categories'].apply(lambda x: 1 if cat in x else 0)

# Get list of DME binary column names
DME_BINARY_COLS = ['dme_' + cat.lower().replace(' ', '_').replace('&', 'and').replace('-', '_') 
                   for cat in all_dme_categories]

print(f"Created {len(DME_BINARY_COLS)} DME binary columns:")
for col in DME_BINARY_COLS:
    print(f"  {col}: {df[col].sum():,} records")


Created 19 DME binary columns:
  dme_glucose_monitors: 44,668 records
  dme_urological_supplies: 26,930 records
  dme_surgical_dressings: 22,598 records
  dme_positive_airway: 902 records
  dme_lower_limb_orthotics: 2,755 records
  dme_parenteral_nutrition: 102 records
  dme_ventilators: 2,205 records
  dme_diabetes_supplies_and_contraceptives: 28,826 records
  dme_oxygen_accessories: 43 records
  dme_oxygen_equipment: 1,759 records
  dme_humidifiers_and_nebulizers: 55 records
  dme_wheelchair_access: 0 records
  dme_pharmacy_dispensing: 19 records
  dme_infusion_pumps: 8,654 records
  dme_inhalation_solutions: 0 records
  dme_breathing_aids: 2,285 records
  dme_hospital_beds: 0 records
  dme_replacement_batteries: 17 records
  dme_tapes_and_medical_supplies: 93 records


In [73]:
# ── 1. Define the feature set ──────────────────────────────────────────────────
# specialty_type, specialty_lvl1, and specialty are now columns on df directly
# (joined in the specialty merge above)  no secondary merge needed here.
# UPDATED: Now includes product rollup columns and DME category binary columns.

ID_COLS = ["record_id", "covered_recipient_npi", "target"]

# Continuous features (will be winsorized + median-imputed).
CONT_FEATURES = [
    "total_amount_of_payment_us_dollars",
    "avg_amount_per_payment",
    "log_total_amount",
    "number_of_payments_included_in_total_amount",
    "payment_to_publication_lag_days",
    "payment_month",
    "payment_quarter",
    "program_year",
]

# Product rollup columns (counts of products by type per record)
# These come from the product pivot created earlier in the notebook
PRODUCT_COLS = [c for c in df.columns if (c.startswith('covered_') or c.startswith('non-covered_')) 
               and c not in ['covered_recipient_type', 'covered_recipient_npi', 
                            'covered_recipient_primary_type_1', 'covered_recipient_license_state_code1',
                            'covered_recipient_license_state_code2', 'covered_recipient_license_state_code3',
                            'covered_recipient_license_state_code4', 'covered_recipient_license_state_code5',
                            'covered_or_noncovered_indicator_1', 'covered_or_noncovered_indicator_2',
                            'covered_or_noncovered_indicator_3', 'covered_or_noncovered_indicator_4',
                            'covered_or_noncovered_indicator_5']]
print(f"Product columns: {PRODUCT_COLS}")

# Binary / indicator features (0/1; median-imputed only, not winsorized).
BIN_FEATURES = [
    "is_weekend_payment",
    "is_q4_payment",
    "is_travel_payment",
    "is_consulting_payment",
    "is_speaker_payment",
    "is_ownership_payment",
    "is_charity_payment",
    "is_third_party_payment",
]

# DME category binary columns (created in the cell above)
# Each column is 1 if the record matches that DME category, 0 otherwise
print(f"DME binary columns: {DME_BINARY_COLS}")

# Categorical columns to label-encode.
# specialty_type / specialty_lvl1 / specialty are already on df from the
# upstream merge  no repeat join required.
CAT_FEATURES_RAW = [
    "nature_short_descr",                   # consulting, food & bev, speaker, etc.
    "form_of_payment_or_transfer_of_value",  # cash vs in-kind
    "covered_recipient_primary_type_1",      # MD / DO / DDS / DPM
    "recipient_state",                       # geographic risk
    "specialty_type",                        # Medical Doctor / Doctor of Dentistry etc.
    "specialty_lvl1",                        # broad specialty group
    "specialty",                             # granular specialty
]
CAT_FEATURES_ENC = [f"{c}_enc" for c in CAT_FEATURES_RAW]

# Combine all features
ALL_FEATURES = CONT_FEATURES + PRODUCT_COLS + BIN_FEATURES + DME_BINARY_COLS + CAT_FEATURES_ENC

print(f"\nTotal features: {len(ALL_FEATURES)}")
print(f"  Continuous: {len(CONT_FEATURES)}")
print(f"  Product: {len(PRODUCT_COLS)}")
print(f"  Binary: {len(BIN_FEATURES)}")
print(f"  DME Binary: {len(DME_BINARY_COLS)}")
print(f"  Categorical (encoded): {len(CAT_FEATURES_ENC)}")

# ── 2. Build the working frame ─────────────────────────────────────────────────
# All columns are already on df  no secondary merge needed.
anomaly_df = df[ID_COLS + CONT_FEATURES + PRODUCT_COLS + BIN_FEATURES + DME_BINARY_COLS + CAT_FEATURES_RAW].copy()

print(f"\nStarting shape: {anomaly_df.shape}")

# ── 3. Label-encode categorical columns ────────────────────────────────────────
# NaNs are filled with sentinel 'UNKNOWN' before encoding so missing categories
# map to a consistent integer rather than raising an error.
for raw_col, enc_col in zip(CAT_FEATURES_RAW, CAT_FEATURES_ENC):
    filled = anomaly_df[raw_col].fillna("UNKNOWN").str.strip().str.upper()
    codes, _ = pd.factorize(filled, sort=True)
    anomaly_df[enc_col] = codes.astype("float64")

# Drop the original string columns  model input must be fully numeric.
anomaly_df.drop(columns=CAT_FEATURES_RAW, inplace=True)

# ── 4. Coerce remaining feature columns to float64 ─────────────────────────────
# pandas nullable Int64 / boolean dtypes must be cast before PyOD / sklearn.
numeric_features = CONT_FEATURES + PRODUCT_COLS + BIN_FEATURES + DME_BINARY_COLS
for col in numeric_features:
    anomaly_df[col] = pd.to_numeric(anomaly_df[col], errors="coerce").astype("float64")

# ── 5. Replace ±Inf with NaN ───────────────────────────────────────────────────
anomaly_df[ALL_FEATURES] = anomaly_df[ALL_FEATURES].replace([np.inf, -np.inf], np.nan)

# ── 6. Report missingness before imputation ────────────────────────────────────
missing_before = anomaly_df[ALL_FEATURES].isna().sum()
print("\nMissing values per feature (before imputation):")
print(missing_before[missing_before > 0].to_string() or "  None")

# ── 7. Median imputation (unsupervised  no label leakage risk) ────────────────
feature_medians = anomaly_df[ALL_FEATURES].median()
anomaly_df[ALL_FEATURES] = anomaly_df[ALL_FEATURES].fillna(feature_medians)

assert anomaly_df[ALL_FEATURES].isna().sum().sum() == 0, (
    "Imputation incomplete  NaNs remain in feature matrix!"
)
print("\nAll NaNs resolved after median imputation.")

# ── 8. Winsorise continuous features at 1st / 99th percentile ─────────────────
# Only winsorize truly continuous features, not binary/count features
WINSOR_LOW, WINSOR_HIGH = 0.01, 0.99

clip_bounds = {}
for col in CONT_FEATURES:
    lo = anomaly_df[col].quantile(WINSOR_LOW)
    hi = anomaly_df[col].quantile(WINSOR_HIGH)
    anomaly_df[col] = anomaly_df[col].clip(lower=lo, upper=hi)
    clip_bounds[col] = (lo, hi)

clip_summary = pd.DataFrame(clip_bounds, index=["p01", "p99"]).T
print("\nWinsorisation bounds (continuous features):")
print(clip_summary.to_string())

# ── 9. Final validation ────────────────────────────────────────────────────────
assert not anomaly_df[ALL_FEATURES].isin([np.inf, -np.inf]).any().any(), (
    "Infinite values found after clipping!"
)
assert anomaly_df[ALL_FEATURES].isna().sum().sum() == 0, (
    "NaN values found after imputation!"
)
print(f"\nFinal anomaly-ready shape: {anomaly_df.shape}")
print("\nFeature dtypes:")
print(anomaly_df[ALL_FEATURES].dtypes.to_string())

# ── 10. Export ─────────────────────────────────────────────────────────────────
anomaly_df.to_csv(
    "/dsa/groups/casestudycf25/team02/silver/cms_general_payments_anomaly_ready.csv",
    index=False,
)
print("\nWritten: cms_general_payments_anomaly_ready.csv")
print(f"  Rows   : {len(anomaly_df):,}")
print(f"  Columns: {anomaly_df.shape[1]}")
print(f"    {len(ID_COLS)} identifiers : {ID_COLS}")
print(f"    {len(CONT_FEATURES)} continuous  : {CONT_FEATURES}")
print(f"    {len(PRODUCT_COLS)} product     : {PRODUCT_COLS}")
print(f"    {len(BIN_FEATURES)} binary      : {BIN_FEATURES}")
print(f"    {len(DME_BINARY_COLS)} DME binary  : {DME_BINARY_COLS}")
print(f"    {len(CAT_FEATURES_ENC)} categorical : {CAT_FEATURES_ENC}")


Product columns: ['covered_biological', 'covered_device', 'covered_drug', 'covered_medical supply', 'non-covered_device', 'non-covered_drug', 'non-covered_medical supply', 'non-covered_unknown']
DME binary columns: ['dme_glucose_monitors', 'dme_urological_supplies', 'dme_surgical_dressings', 'dme_positive_airway', 'dme_lower_limb_orthotics', 'dme_parenteral_nutrition', 'dme_ventilators', 'dme_diabetes_supplies_and_contraceptives', 'dme_oxygen_accessories', 'dme_oxygen_equipment', 'dme_humidifiers_and_nebulizers', 'dme_wheelchair_access', 'dme_pharmacy_dispensing', 'dme_infusion_pumps', 'dme_inhalation_solutions', 'dme_breathing_aids', 'dme_hospital_beds', 'dme_replacement_batteries', 'dme_tapes_and_medical_supplies']

Total features: 50
  Continuous: 8
  Product: 8
  Binary: 8
  DME Binary: 19
  Categorical (encoded): 7

Starting shape: (932908, 53)

Missing values per feature (before imputation):
Series([], )

All NaNs resolved after median imputation.

Winsorisation bounds (continuou

In [74]:
# Quick sanity check: verify the exported file is model-ready
anomaly_check = pd.read_csv(
    "/dsa/groups/casestudycf25/team02/silver/cms_general_payments_anomaly_ready.csv"
)

feature_cols = [c for c in anomaly_check.columns if c not in ["record_id", "covered_recipient_npi", "target"]]

nan_count  = anomaly_check[feature_cols].isna().sum().sum()
inf_count  = np.isinf(anomaly_check[feature_cols].values).sum()
non_numeric = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(anomaly_check[c])]

print("=" * 50)
print("Anomaly-Ready Dataset Validation")
print("=" * 50)
print(f"  Shape            : {anomaly_check.shape}")
print(f"  NaN values       : {nan_count}" if nan_count == 0 else f"  NaN values       : {nan_count} ")
print(f"  Inf values       : {inf_count}" if inf_count == 0 else f"  Inf values       : {inf_count}")
print(f"  Non-numeric cols : {non_numeric or 'None'}" if not non_numeric else f"  Non-numeric cols : {non_numeric}  ")
print(f"  Target balance   : {anomaly_check['target'].value_counts().to_dict()}")
print("\nFeature summary:")
anomaly_check[feature_cols].describe().T[["min", "max", "mean", "50%"]]


Anomaly-Ready Dataset Validation
  Shape            : (932908, 53)
  NaN values       : 0
  Inf values       : 0
  Non-numeric cols : None
  Target balance   : {0: 932441, 1: 467}

Feature summary:


,min,max,mean,50%
total_amount_of_payment_us_dollars,2.470000,4500.000000,180.188599,24.330000
avg_amount_per_payment,2.470000,4500.000000,178.692565,24.280000
log_total_amount,1.244155,8.412055,3.687727,3.231989
number_of_payments_included_in_total_amount,1.000000,1.000000,1.000000,1.000000
payment_to_publication_lag_days,563.000000,1616.000000,1047.527410,1025.000000
payment_month,1.000000,12.000000,6.706504,7.000000
payment_quarter,1.000000,4.000000,2.558351,3.000000
program_year,2021.000000,2023.000000,2022.112289,2022.000000
covered_biological,0.000000,2.000000,0.000656,0.000000
covered_device,0.000000,5.000000,1.137182,1.000000


In [75]:
df.to_csv("/dsa/groups/casestudycf25/team02/silver/cms_general_payments_clean.csv", index=False)

In [76]:
df.to_csv("cms_general_payments_clean.csv", index=False)

In [ ]:
tempdf = df[:30000].copy()
tempdf.to_csv("sample_general_payments_clean.csv", index = False)

In [ ]:
# df_filtered.to_csv("/dsa/groups/casestudycf25/team02/cms_general_payments_clean.csv.gz", index=False, compression="gzip")
# df_filtered.to_csv("cms_general_payments_clean.csv.gz", index=False, compression="gzip")

# import math

# rows_per_file = 1_000_000
# n_rows = len(df)
# n_files = math.ceil(n_rows / rows_per_file)

# for i in range(n_files):
#     start = i * rows_per_file
#     end = start + rows_per_file

#     outname = f"cms_general_payments_clean_part_{i+1}.csv.gz"
#     df.iloc[start:end].to_csv(outname, index=False, compression="gzip")

#     print(f"Created {outname}")
# df.shape

In [ ]:
# Run locally
# import pandas as pd
# import glob

# # Adjust the path/pattern to match files
# files = glob.glob("raw_data/cms_general_payments_cleaned/cms_general_payments_clean_part_*.csv")

# df_list = []

# for f in files:
#     print("Reading:", f)
#     df_list.append(pd.read_csv(f, low_memory=False))

# # Combine all into one big dataframe
# df = pd.concat(df_list, ignore_index=True)